# Workshop Agent 5 — The Verification / Fact-Checking Agent

**Book:** *Agents* by Imran Ahmad (Packt Publishing, 2026)  
**Chapter:** 8 — Data Analysis and Reasoning Agents (§8.2, pp. 211–215; §8.4, pp. 220–226)

In this workshop you will build a journalism-grade Verification and Validation agent — a system that does not merely generate answers but defends them. Given a news statement, the agent decomposes it into discrete, verifiable claims, retrieves evidence against authoritative data, and reasons about conflicting evidence to assign each claim a verdict — `Confirmed`, `Mostly True`, `Contradicted`, or `Unverified` — always returned with its supporting evidence and provenance.

You will first examine the agent's conceptual foundation (§8.2): fact-checking, logical coherence, retrieval-augmented evaluation, and consistency analysis, including a natural language inference (NLI) demonstration for handling conflicting evidence. You will then implement the complete newsroom fact-checking case study (§8.4): a **Claim Extractor**, an **Evidence Retriever**, and a tolerance-based **Verifier** that together produce an editorial report.

The notebook runs end-to-end in **Simulation Mode** — no API key is required. Every LLM call routes through `llm_call()` with a deterministic, chapter-accurate mock fallback, and the NLI demo degrades gracefully to precomputed chapter scores when `transformers`/`torch` are unavailable.


In [ ]:
# Google Colab bootstrap — runs only on Colab, no-op everywhere else.
# Locally you are already inside the agent folder with requirements installed.
import os
import sys

if "google.colab" in sys.modules:
    AGENT_DIR = "05-verification-fact-checking-agent"
    if not os.path.exists("/content/repo"):
        os.system("git clone --depth 1 https://github.com/cloudanum/ws-10-agents /content/repo")
    os.chdir(f"/content/repo/{AGENT_DIR}")
    # Keep Colab's preinstalled scientific/kernel stack: the kernel already has
    # numpy, pandas, pydantic and ipykernel loaded, so letting pip replace them
    # (e.g. building numpy 1.26.4 from source or upgrading ipykernel) breaks the
    # running kernel with ABI errors or an OOM kill. Filter those lines out of
    # requirements and constrain the rest of the install to the installed versions.
    import re
    from importlib.metadata import PackageNotFoundError, version
    filtered = [
        line for line in open("requirements.txt")
        if not re.match(r"\s*(numpy|pandas|pydantic|jupyter|ipykernel)\b", line, re.IGNORECASE)
    ]
    with open("/tmp/colab_requirements.txt", "w") as fh:
        fh.writelines(filtered)
    pins = []
    for pkg in ("numpy", "pandas", "pydantic", "ipykernel"):
        try:
            pins.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            pass
    with open("/tmp/colab_constraints.txt", "w") as fh:
        fh.write("\n".join(pins) + "\n")
    import subprocess
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", "/tmp/colab_requirements.txt",
         "--constraint", "/tmp/colab_constraints.txt"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("pip install failed — re-running without -q for the full resolver report:\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "-r", "/tmp/colab_requirements.txt",
             "--constraint", "/tmp/colab_constraints.txt"],
        )
        raise RuntimeError("Colab bootstrap: pip install failed (see resolver report above)")
    print(f"Colab setup complete — working directory: {os.getcwd()}")
else:
    print("Not on Colab — skipping bootstrap (local setup already in place).")

---
## Section 0 — Environment Setup

This cell imports the shared utilities, resolves the API key through a
three-tier cascade (`.env` → `os.getenv` → `getpass`), and sets the global
`SIMULATION_MODE` flag. If no key is found, the notebook activates Simulation
Mode automatically.

In [1]:
# ── Section 0: Environment Setup ─────────────────────────────
# Ref: Strategy §4.1 — Three-tier API key resolution

import sys, os, json, re, textwrap
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for utils imports
if '.' not in sys.path:
    sys.path.insert(0, '.')

from utils import load_api_key, log, MockLLM, llm_call, fail_gracefully

# ── Resolve API key ──────────────────────────────────────────
API_KEY, SIMULATION_MODE = load_api_key()

client = None
if not SIMULATION_MODE:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
        log.success('OpenAI client initialized. Running in LIVE mode.')
    except Exception as e:
        log.error(f'OpenAI client init failed: {e}')
        SIMULATION_MODE = True
        log.info('Falling back to SIMULATION MODE.')

if SIMULATION_MODE:
    log.info('No API key detected. Running in SIMULATION MODE.')
    log.info('All outputs are chapter-accurate mocks. See AGENTS.md.')

[INFO 18:22:23] No API key detected. Running in SIMULATION MODE.
[INFO 18:22:23] All outputs are chapter-accurate mocks. See AGENTS.md.


---
## Section 2 — The Verification and Validation Agent

**Chapter Ref:** §8.2 (pp. 211–215)

While statistical reasoning allows agents to draw inferences from data,
every analytical system must also be able to verify and defend its conclusions.
The Verification and Validation (V&V) agent addresses this by ensuring that
generated insights, predictions, and explanations remain trustworthy, factual,
and logically coherent.

A V&V agent performs four essential functions:

### 2.1 — Fact-Checking (§8.2.1, pp. 211–212)

When an analytical agent produces a statement such as *"Revenue increased by
12% last quarter"*, the V&V agent retrieves corresponding records from verified
data sources, performs independent calculations, and compares findings to the
claim. The agent decomposes statements into discrete, verifiable claims and
tests each as a hypothesis against reliable evidence.

### 2.2 — Logical Coherence (§8.2.2, pp. 212–213)

The agent evaluates whether reasoning steps follow valid inference patterns.

> **📝 Note — Common Logical Errors (p. 212)**  
> Several categories of logical error recur in agent-generated analyses: **Premise–conclusion mismatches** (concluding revenue increased after noting a decline in unit sales), **Circular reasoning** (an intermediate step reuses the claim it's trying to prove), and **Scope violations** (a finding qualified for one segment is generalized to the entire population). The agent can decompose reasoning chains into directed graphs of claims and check each edge for valid inference.

### 2.3 — Retrieval-Augmented Evaluation (§8.2.3, p. 213)

Combines the interpretive flexibility of language models with deterministic
verification techniques — retrieving historical records, recalculating figures
independently, and checking alignment with conclusions.

### 2.4 — Consistency Analysis (§8.2.4, pp. 213–214)

In multi-agent systems, V&V agents perform structured post-processing:
length/formatting checks, structural validations, rule-based constraints,
adversarial red-teaming, and human-in-the-loop triggers.


### Section 2.5 — Handling Conflicting Evidence: NLI Demo

**Chapter Ref:** §8.2.5 (pp. 214–215)

The code below demonstrates NLI using `facebook/bart-large-mnli` (~1.6 GB).
If the model or its dependencies (`transformers`, `torch`) are unavailable,
the notebook falls back to precomputed scores matching the chapter's Q4 profit
claim example.

**Premise:** *"In Q4, the company reported a 15% year-over-year increase in
net profit."*  
**Hypothesis:** *"The company's profits increased by 15% last quarter."*

> **📝 Note — NLI Classification Labels (p. 212)**  
> NLI models classify the relationship between evidence and a claim as one of three outcomes: **Supports (Entailment)** — the evidence confirms the claim; **Refutes (Contradiction)** — the evidence disproves the claim; **Neutral** — the evidence is inconclusive or unrelated. Advanced V&V agents balance evidence across multiple sources, weight their credibility, and synthesize results into a confidence score.


In [2]:
# ── Section 2.5: NLI Demo (BART-MNLI) ──────────────────────
# Ref: §8.2.5 — Handling Conflicting Evidence
# Fallback: precomputed scores per Strategy §4.4

# Premise and hypothesis from chapter §8.2.5
premise = "In Q4, the company reported a 15% year-over-year increase in net profit."
hypothesis = "The company's profits increased by 15% last quarter."

nli_scores = None

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch

    log.info('Loading BART-MNLI model (facebook/bart-large-mnli)...', section='8.2.5')

    tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-mnli')
    nli_model = AutoModelForSequenceClassification.from_pretrained('facebook/bart-large-mnli')
    id2label = nli_model.config.id2label

    inputs = tokenizer(premise, hypothesis, return_tensors='pt', truncation=True)

    with torch.no_grad():
        logits = nli_model(**inputs).logits[0]
        probs = torch.softmax(logits, dim=-1).tolist()

    nli_scores = {id2label[i].lower(): round(probs[i], 4) for i in range(len(probs))}
    log.success(f'Live NLI inference complete: {nli_scores}', section='8.2.5')

except (ImportError, OSError, Exception) as e:
    log.error(
        f'BART-MNLI model unavailable ({type(e).__name__}). '
        'Displaying precomputed results.',
        section='8.2.5'
    )
    # Precomputed fallback — matches chapter's Q4 profit example
    nli_scores = {'entailment': 0.92, 'neutral': 0.05, 'contradiction': 0.03}
    log.info(f'Precomputed NLI scores: {nli_scores}', section='8.2.5')

# ── Display results ─────────────────────────────────────────
print(f'\nPremise:    "{premise}"')
print(f'Hypothesis: "{hypothesis}"')
print(f'\nNLI Scores: {nli_scores}')

if nli_scores:
    best = max(nli_scores, key=nli_scores.get)
    log.success(
        f'Verdict: {best.upper()} (confidence: {nli_scores[best]:.2%})',
        section='8.2.5'
    )

[HANDLED ERROR 18:22:23 §8.2.5] BART-MNLI model unavailable (ModuleNotFoundError). Displaying precomputed results.
[INFO 18:22:23 §8.2.5] Precomputed NLI scores: {'entailment': 0.92, 'neutral': 0.05, 'contradiction': 0.03}

Premise:    "In Q4, the company reported a 15% year-over-year increase in net profit."
Hypothesis: "The company's profits increased by 15% last quarter."

NLI Scores: {'entailment': 0.92, 'neutral': 0.05, 'contradiction': 0.03}
[SUCCESS 18:22:23 §8.2.5] Verdict: ENTAILMENT (confidence: 92.00%)


---
## Section 4 — Case Study: News Fact-Checking Assistant

**Chapter Ref:** §8.4 (pp. 220–226) — An Agent for Journalistic Integrity

A major newsroom deployed a Verification and Validation agent to assist
editors during high-pressure events where speed and accuracy must coexist.
The agent is composed of three cooperating components:

- **Claim Extractor** — Scans text for verifiable numerical statements and
  converts them into structured records with metric, value, entity, and period.
- **Evidence Retriever** — Queries curated official sources to obtain
  authoritative values with provenance.
- **Verifier** — Compares claimed and authoritative values with defined
  tolerances, then assigns a label: `Confirmed`, `Mostly True`, `Contradicted`,
  or `Unverified`.

> **📝 Note — Tolerance Thresholds (p. 223)**  
> For percentage-based claims, the system applies a difference threshold of 0.5 percentage points. For monetary values, it applies a threshold of $500,000 to absorb common rounding in public communications. In production, calibrate tolerances by beat and metric.

> **📝 Note — Operational Results (p. 226)**  
> Deploying this agent reduced verification time from hours to minutes. Editors retained control over the narrative while trusting that quantitative claims were consistent with official data. For production use, replace the in-memory store with your data platform, introduce freshness checks, and route low-confidence outcomes to a human review queue.


In [3]:
# ── Section 4: Trusted Database & Article Text ──────────────
# Ref: §8.4 — Authoritative data store and test article
#       (verbatim from chapter)

from typing import List, Dict, Any

# ── Authoritative data: The trusted internal database ───────
# In production, this would be a warehouse or official API with
# access control, versioning, and data freshness policies.

trusted_database: Dict[str, Dict[str, Any]] = {
    "ottawa_unemployment_rate_change_2024": {
        "value": -0.048,
        "source": "Statistics Canada, Labour Force Survey, Table 14-10-0287-01",
        "notes": "Annual change from 2023 to 2024 for Ottawa--Gatineau CMA."
    },
    "city_budget_surplus_2024": {
        "value": 15_200_000,
        "source": "City of Ottawa Annual Financial Report 2024",
        "notes": "Reported surplus for the fiscal year ending 2024."
    }
}

article_text = (
    "A new report on Ottawa's economy shows promising signs of recovery. "
    "According to official city documents, the city's unemployment rate "
    "fell by 5% last year, a significant improvement driven by the tech "
    "sector. Furthermore, the municipal government reported a budget "
    "surplus of $12 million for the 2024 fiscal year."
)

log.info('Trusted database loaded (2 entries).', section='8.4')
log.info(f'Article text loaded ({len(article_text)} chars).', section='8.4')

[INFO 18:22:23 §8.4] Trusted database loaded (2 entries).
[INFO 18:22:23 §8.4] Article text loaded (314 chars).


In [4]:
# ── Section 4: Claim Extraction ─────────────────────────────
# Ref: §8.4 — extract_claims_from_text()
# LLM-first with regex fallback, verbatim from chapter.
# Mock key: "claim_extraction"

@fail_gracefully(fallback_value=[], section='8.4')
def extract_claims_from_text(text: str) -> List[Dict[str, Any]]:
    """Extract verifiable numeric claims from article text.

    Uses llm_call() with context_key='claim_extraction' for robust
    parsing. If the LLM path fails, a regex fallback keeps the
    workflow operational.

    Returns claims with fields: claim_text, metric, value, entity, period.

    Ref: §8.4 — Claim Extraction (LLM-first with a safe fallback)
    """
    system_prompt = (
        "You are an expert fact-checker. Extract verifiable numeric claims "
        "from the text. For each claim, return claim_text, metric, value, "
        "entity, period as a JSON object in a JSON field called 'claims' "
        "which is a list."
    )
    user_prompt = f'Extract claims from: {text}'

    raw = llm_call(
        system_prompt, user_prompt,
        context_key='claim_extraction',
        simulation_mode=SIMULATION_MODE,
        client=client,
    )

    # ── Parse LLM / mock response ───────────────────────────
    try:
        data = json.loads(raw)
        claims = data.get('claims', [])
        for c in claims:
            c['claim_text'] = str(c.get('claim_text', '')).strip()
            c['metric']     = str(c.get('metric', '')).strip()
            c['value']      = str(c.get('value', '')).strip()
            c['entity']     = str(c.get('entity', '')).strip()
            c['period']     = str(c.get('period', '')).strip()
        if claims:
            return claims
    except (json.JSONDecodeError, AttributeError):
        log.error('LLM response not valid JSON; trying regex fallback.', section='8.4')

    # ── Regex fallback (from chapter) ───────────────────────
    claims = []
    m1 = re.search(r'(unemployment.*?fell by\s+(-?\d+(\.\d+)?)\s*%)', text, re.I)
    if m1:
        claims.append({
            'claim_text': m1.group(1),
            'metric': 'unemployment rate change',
            'value': f'{m1.group(2)}%',
            'entity': 'Ottawa',
            'period': '2024'
        })

    m2 = re.search(
        r'budget surplus of\s*\$?\s*([0-9]+(\.\d+)?)\s*(million)?\s*for the\s*(\d{4})',
        text, re.I
    )
    if m2:
        amount = float(m2.group(1)) * (1_000_000 if m2.group(3) else 1)
        claims.append({
            'claim_text': m2.group(0),
            'metric': 'budget surplus',
            'value': str(amount),
            'entity': 'Ottawa',
            'period': m2.group(4)
        })

    return claims


# ── Run extraction ──────────────────────────────────────────
claims = extract_claims_from_text(article_text)
log.success(f'Extracted {len(claims)} claim(s).', section='8.4')
for i, c in enumerate(claims, 1):
    print(f'  Claim {i}: {c["claim_text"]}')
    print(f'           metric={c["metric"]}, value={c["value"]}, '
          f'entity={c["entity"]}, period={c["period"]}')

[INFO 18:22:23 §8.4] Entering extract_claims_from_text()
[INFO 18:22:23 §8.4] MockLLM serving 'claim_extraction' (Section 8.4)
[SUCCESS 18:22:23 §8.4] extract_claims_from_text() completed.
[SUCCESS 18:22:23 §8.4] Extracted 2 claim(s).
  Claim 1: the city's unemployment rate fell by 5% last year
           metric=unemployment rate change, value=5%, entity=Ottawa, period=2024
  Claim 2: budget surplus of $12 million for the 2024 fiscal year
           metric=budget surplus, value=12000000, entity=Ottawa, period=2024


In [5]:
# ── Section 4: Mapping, Parsing, and Verification ───────────
# Ref: §8.4 — _map_to_db_key(), _parse_percentage(),
#       verify_claim() — verbatim from chapter.
# Tolerances: 0.5 pp for percentages, $500K for monetary.

def _map_to_db_key(metric: str, entity: str, period: str) -> str:
    """Map a claim's fields to a canonical trusted_database key.

    Acts as a simplified Evidence Retriever: resolves each claim
    to an authoritative record.

    Ref: §8.4 — Mapping, parsing, and verification
    """
    m = (metric or '').lower()
    e = (entity or '').lower()
    p = str(period or '')
    if 'unemployment' in m and 'ottawa' in e and '2024' in p:
        return 'ottawa_unemployment_rate_change_2024'
    if 'budget surplus' in m and 'ottawa' in e and '2024' in p:
        return 'city_budget_surplus_2024'
    return ''


def _parse_percentage(value: str) -> float:
    """Parse a percentage string into a decimal.

    Ref: §8.4 — _parse_percentage()
    """
    v = value.replace(' ', '')
    return float(v[:-1]) / 100 if v.endswith('%') else float(v)


@fail_gracefully(
    fallback_value={'status': 'Error', 'details': 'Verification unavailable'},
    section='8.4'
)
def verify_claim(claim: Dict[str, Any]) -> Dict[str, Any]:
    """Verify a single claim against the trusted database.

    Applies tolerance-based comparison:
    - Percentage claims: 0.5 percentage-point threshold
    - Monetary claims: $500,000 threshold

    Returns a verdict dict with status, details, source, and notes.

    Ref: §8.4 — verify_claim()
    """
    db_key = _map_to_db_key(
        claim.get('metric', ''),
        claim.get('entity', ''),
        claim.get('period', '')
    )
    if db_key not in trusted_database:
        return {
            'claim': claim.get('claim_text', ''),
            'status': 'Unverified',
            'details': 'No matching internal data source.'
        }

    evidence = trusted_database[db_key]
    actual = evidence['value']
    claimed_raw = claim.get('value', '')

    try:
        if isinstance(claimed_raw, str) and '%' in claimed_raw:
            claimed = _parse_percentage(claimed_raw)
            diff = abs(claimed - actual)
            tol = 0.005  # 0.5 percentage points
            status = (
                'Confirmed' if diff == 0
                else ('Mostly True' if diff <= tol else 'Contradicted')
            )
            details = (
                f'Claimed {claimed_raw}, actual '
                f'{round(actual * 100, 2)}% '
                f'(\u0394={round(diff * 100, 2)} pp)'
            )
        else:
            claimed = float(claimed_raw)
            diff = abs(claimed - actual)
            tol = 500_000  # $500K
            status = (
                'Confirmed' if diff == 0
                else ('Mostly True' if diff <= tol else 'Contradicted')
            )
            details = (
                f'Claimed ${int(claimed):,}, actual '
                f'${int(actual):,} '
                f'(\u0394=${int(diff):,})'
            )
    except Exception:
        return {
            'claim': claim.get('claim_text', ''),
            'status': 'Error',
            'details': f'Could not parse value {claimed_raw}'
        }

    return {
        'claim': claim.get('claim_text', ''),
        'metric': claim.get('metric', ''),
        'entity': claim.get('entity', ''),
        'period': claim.get('period', ''),
        'status': status,
        'details': details,
        'source': evidence['source'],
        'notes': evidence.get('notes', '')
    }


log.info('Verification functions defined.', section='8.4')

[INFO 18:22:23 §8.4] Verification functions defined.


In [6]:
# ── Section 4: Orchestration & Editorial Report ─────────────
# Ref: §8.4 — Orchestration, reporting, and demonstration

log.info('Starting Fact-Check Agent...', section='8.4')

if not claims:
    log.error('No verifiable numeric claims found.', section='8.4')
else:
    log.info(f'Found {len(claims)} claim(s). Verifying...', section='8.4')
    print('-' * 72)

    for c in claims:
        r = verify_claim(c)

        # Color-code by verdict
        claim_text = r.get('claim', '')
        status = r.get('status', 'Unknown')
        details = r.get('details', '')

        if status in ('Confirmed', 'Mostly True'):
            log.success(f'[{status}] {claim_text}', section='8.4')
        else:
            log.error(f'[{status}] {claim_text}', section='8.4')

        print(f'  Details: {details}')
        if r.get('source'):
            print(f'  Source:  {r["source"]}')
        if r.get('notes'):
            print(f'  Notes:   {r["notes"]}')
        print('-' * 72)

log.success('Fact-Check Agent complete. Editorial report above.', section='8.4')

[INFO 18:22:23 §8.4] Starting Fact-Check Agent...
[INFO 18:22:23 §8.4] Found 2 claim(s). Verifying...
------------------------------------------------------------------------
[INFO 18:22:23 §8.4] Entering verify_claim()
[SUCCESS 18:22:23 §8.4] verify_claim() completed.
[HANDLED ERROR 18:22:23 §8.4] [Contradicted] the city's unemployment rate fell by 5% last year
  Details: Claimed 5%, actual -4.8% (Δ=9.8 pp)
  Source:  Statistics Canada, Labour Force Survey, Table 14-10-0287-01
  Notes:   Annual change from 2023 to 2024 for Ottawa--Gatineau CMA.
------------------------------------------------------------------------
[INFO 18:22:23 §8.4] Entering verify_claim()
[SUCCESS 18:22:23 §8.4] verify_claim() completed.
[HANDLED ERROR 18:22:23 §8.4] [Contradicted] budget surplus of $12 million for the 2024 fiscal year
  Details: Claimed $12,000,000, actual $15,200,000 (Δ=$3,200,000)
  Source:  City of Ottawa Annual Financial Report 2024
  Notes:   Reported surplus for the fiscal year ending 2024

---
## Summary — Workshop Agent 5

- **Claim extraction** — An LLM-first extractor (with a regex fallback) converts article text into structured, verifiable claims: metric, value, entity, period.
- **Evidence retrieval** — Each claim is mapped to an authoritative record in the trusted database, carrying full source provenance.
- **Tolerance-based verification** — Percentage claims are checked against a 0.5 percentage-point threshold and monetary claims against a $500K threshold, yielding `Confirmed`, `Mostly True`, `Contradicted`, or `Unverified` verdicts.
- **Conflicting evidence** — NLI scoring (§8.2.5) classifies the evidence–claim relationship as entailment, neutral, or contradiction; Simulation Mode uses chapter-accurate precomputed scores.
- **Graceful degradation** — Every tool/LLM call is wrapped in `@fail_gracefully` with color-coded logging, so the agent never hard-fails.

**Further reading:** *Agents* by Imran Ahmad (Packt, 2026), Chapter 8 — §8.2 *The Verification and Validation Agent* (pp. 211–215) and §8.4 *Case Study: News Fact-Checking Assistant* (pp. 220–226).
